# Chorale Rule Testing Harness

This notebook is a thin front end for testing rule costs on a four-part Bach chorale with figured bass. Most of the mechanics live in Python modules under `harmonizer/`; the main editable surface here is the rulebook cell.

In [ ]:
# Setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

from harmonizer import *
from harmonizer.notebook_display import display_cost_summary, export_score_preview_html, render_score_inline


## Pick And Display A Chorale

This renders the original MusicXML directly, preserving the figured bass and adding no generated or alternate-voice overlay.

In [ ]:
CHORALE_ID = "BWV_33.06"  #@param {type:"string"}
CHORALE_ID = "BWV_244.03"
VEROVIO_SCALE = 34  #@param {type:"integer"}

musicxml_path = resolve_bcfb_musicxml_path(CHORALE_ID, workspace_root=PROJECT_ROOT)
print(f"MusicXML: {musicxml_path}")
render_score_inline(musicxml_path, scale=VEROVIO_SCALE, page_width=1800)


## Rule Format

There are currently two rule styles in the repo:

- The declarative `rulebook(...)` DSL below is the recommended format for this harness. It stores rules as data with stable `kind` names and parameters, which means the notebook, scorer, and future generator can all inspect the same rule set.
- The code-first `RuleSet` API lets you write Python methods directly against `VerticalEvent` objects. That is useful for experiments, but it is less portable and harder to turn into generator controls.

For now, edit the DSL cell below. The current `cover_chord_tones(...)` rule is role-aware and can override costs by inversion: omitted fifths are cheaper than omitted thirds, chromatic spellings like C# can satisfy an inferred C-letter chord tone, and unfigured default `5/3` events are downweighted because they often catch passing bass motion. The current `prefer_range(...)` rule has a free semitone band around each center, so ordinary Bach alto/tenor placements do not swamp the rule-cost view.

In [ ]:
# Rule DSL
RULES = rulebook(
    "Starter Chorale Constraints",
    cover_chord_tones(
        root=500,
        third=500,
        fifth=80,
        seventh=300,
        ninth=150,
        explicit=500,
        other=150,
        unfigured_scale=0.25,
        chromatic_same_letter_ok=True,
        notes=(
            "Role-aware coverage: roots/thirds matter most, fifths are cheap to omit, "
            "and unfigured default 5/3 events are downweighted because they often behave "
            "like passing bass motion in this corpus."
        ),
    ),
    avoid_voice_crossing(
        penalty=100,
        allow_equal=True,
        notes="Equality is allowed. Only strict crossing is penalized.",
    ),
    prefer_range(
        alto="C4",
        tenor="G3",
        cost_per_semitone=3,
        free_semitones=10,
        notes=(
            "Soft tessitura nudge: ordinary Bach alto/tenor placements should not "
            "dominate the debug costs."
        ),
    ),
    prefer_doublings(
        root=0,
        fifth=20,
        third=50,
        seventh=70,
    ),
    prefer_melodic_intervals(
        voices=[A, T],
        unison=0,
        step=0,
        third=5,
        fourth=10,
        fifth=20,
        sixth=35,
        seventh=50,
        octave_or_larger=70,
    ),
    recover_leaps(
        voices=[A, T],
        leap_gt="third",
        require="step_opposite",
        penalty=10,
    ),
    prepare_leaps(
        voices=[A, T],
        leap_gt="third",
        require="step_opposite",
        penalty=10,
        enabled=False,
        notes="Present for shape-testing, but disabled by default because this rule is still underspecified.",
    ),
    avoid_parallel_perfects(
        pairs=ALL_VOICE_PAIRS,
        fifth=100,
        octave=100,
        unison=100,
    ),
    avoid_similar_motion_into_perfects(
        pairs=[(S, B)],
        fifth=50,
        octave=50,
        unison=50,
        require_upper_leap=True,
        notes=(
            "Direct perfects are tracked by default only in the outer voices, and only "
            "when the upper voice leaps."
        ),
    ),
    prefer_independent_motion(
        pairs=ALL_VOICE_PAIRS,
        contrary=0,
        oblique=0,
        similar=5,
        ignore_if_perfect_rule_applies=True,
    ),
    description="Notebook-editable starter rule set for rule and generator debugging.",
)

#print(RULES.to_json())


## Analyze Rule Costs

`hybrid` matches the current two-stream model: harmonic and motion rules run on anchor events, while melodic/range/leap rules still run on the raw event stream. Use `anchors` to score only anchor beats, or `all` to score every raw event as a harmonic checkpoint.

In [ ]:
SCORE_SCOPE = "hybrid"  #@param ["hybrid", "anchors", "all"]
ANCHOR_THRESHOLD = 0.45  #@param {type:"number"}
TOP_EVENTS = 12  #@param {type:"integer"}

analysis = analyze_musicxml_rules(
    musicxml_path,
    RULES,
    anchor_threshold=ANCHOR_THRESHOLD,
    score_scope=SCORE_SCOPE,
)

raw_csv = write_cost_rows_csv(
    ARTIFACTS_DIR / f"{CHORALE_ID}_rule_harness_costs.csv",
    analysis,
)
anchor_csv = write_cost_rows_csv(
    ARTIFACTS_DIR / f"{CHORALE_ID}_rule_harness_anchor_costs.csv",
    analysis,
    rows=[row for row in analysis.rows if row["is_anchor"] == "yes"],
)

print(f"Rows: {len(analysis.rows)} raw events, {len(analysis.anchor_indices)} anchors")
print(f"CSV: {raw_csv}")
print(f"Anchor CSV: {anchor_csv}")
display_cost_summary(analysis, top_events=TOP_EVENTS, anchors_only=(SCORE_SCOPE == "anchors"))


## Render Cost Highlights

The highlighted score is a copy of the original MusicXML. The highest-cost events are colored from deep red to lighter red/orange. Hover over highlighted notes to see the event location and nonzero rule costs; the table above is the fallback if a browser does not expose SVG tooltips cleanly.

In [ ]:
HIGHLIGHT_EVENTS = 16  #@param {type:"integer"}
MINIMUM_HIGHLIGHT_COST = 1.0  #@param {type:"number"}
HIGHLIGHT_ANCHORS_ONLY = False  #@param {type:"boolean"}

highlighted = write_cost_highlighted_musicxml(
    musicxml_path,
    ARTIFACTS_DIR / f"{CHORALE_ID}_rule_harness_highlight.musicxml",
    analysis,
    max_events=HIGHLIGHT_EVENTS,
    minimum_cost=MINIMUM_HIGHLIGHT_COST,
    anchors_only=HIGHLIGHT_ANCHORS_ONLY,
)
preview_path = export_score_preview_html(
    highlighted.path,
    ARTIFACTS_DIR / f"{CHORALE_ID}_rule_harness_highlight_preview.html",
    title=f"Rule Cost Highlights: {CHORALE_ID}",
    scale=VEROVIO_SCALE,
    page_width=1800,
    tooltips_by_color=highlighted.tooltips_by_color,
)

print(f"Highlighted MusicXML: {highlighted.path}")
print(f"Preview HTML: {preview_path}")
print(f"Highlighted event indices: {highlighted.highlighted_event_indices}")
render_score_inline(
    highlighted.path,
    scale=VEROVIO_SCALE,
    page_width=1800,
    tooltips_by_color=highlighted.tooltips_by_color,
)
